# 3장 2강: 규제 — L1(Lasso), L2(Ridge), ElasticNet

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 Ridge, Lasso, ElasticNet 규제가 회귀계수와 일반화 성능에 어떤 영향을 주는지 확인합니다.

- 규제가 계수의 크기를 줄이는 이유를 이해합니다.
- `alpha` 변화에 따라 Ridge의 계수 크기와 Test 성능이 어떻게 달라지는지 확인합니다.
- Lasso가 일부 계수를 정확히 0으로 만드는지 확인합니다.
- ElasticNet이 L1과 L2를 함께 사용하는 모델이라는 점을 확인합니다.
- Ridge / Lasso / ElasticNet의 계수와 Test 성능을 비교합니다.

> 이 실습에서는 3장 2강 교안에서 다룬 규제, alpha, Lasso, Ridge, ElasticNet, StandardScaler, R² 비교만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

규제에 따른 계수 변화를 비교하기 위해 결측치가 없는 숫자형 특성 중 다음 12개를 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

규제는 계수의 크기를 기준으로 벌점을 주기 때문에 특성의 스케일을 맞추는 과정이 중요합니다.

이번 실습에서는 다음 순서로 데이터를 준비합니다.

1. Ames Housing 데이터 불러오기
2. 사용할 특성과 `SalePrice` 선택
3. Train / Test 분리
4. `StandardScaler`를 이용한 특성 표준화

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train 크기:", X_train_scaled.shape)
print("Test 크기:", X_test_scaled.shape)

Train 크기: (1168, 12)
Test 크기: (292, 12)


---

# 필수 1. Ridge와 alpha에 따른 계수 축소

## 배경

Ridge는 **L2 규제**를 사용합니다.

Ridge는 계수의 제곱합에 벌점을 주기 때문에 큰 계수를 더 강하게 억제하고, 모든 계수를 0에 가깝게 줄이는 특징이 있습니다.

## 문제 1. 여러 alpha 값에서 Ridge를 비교하세요.

### 요구사항

다음 alpha 값을 사용합니다.

```python
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
```

각 alpha에 대해 다음을 수행합니다.

1. `Ridge(alpha=alpha)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test 데이터를 예측합니다.
4. Test R²를 계산합니다.
5. `abs(model.coef_).sum()`으로 전체 계수 크기를 확인합니다.
6. 결과를 DataFrame으로 정리합니다.

### 확인 포인트

- alpha가 커질수록 규제 강도가 강해지는가?
- alpha가 커질수록 전체 계수 크기가 작아지는가?
- alpha가 너무 커지면 Test 성능이 낮아질 수 있는가?

In [2]:
# TODO
# 여러 alpha 값에 대해 Ridge의 Test R²와 계수 크기를 비교하세요.


# Ridge(L2 규제)

alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

# 1. 결과 저장 리스트
results = []

# 2. alpha별 Ridge 학습 및 평가
for alpha in alpha_values:
    
    # 학습
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)

    # 예측
    y_pred = model.predict(X_test_scaled)

    test_r2 = r2_score(y_test, y_pred)

    coef_sum = abs(model.coef_).sum()

    results.append({
        "alpha": alpha,
        "Test_R2": test_r2,
        "Coef_Abs_Sum": coef_sum
    })

# 3. DataFrame 생성
result_df = pd.DataFrame(results).round(4)

display(result_df)


,alpha,Test_R2,Coef_Abs_Sum
0,0.001,0.7994,102172.5908
1,0.010,0.7994,102172.4637
2,0.100,0.7994,102171.0968
3,1.000,0.7993,102149.8317
4,10.000,0.7991,101725.4315
5,100.000,0.7956,97390.1202


- 실행결과

|alpha|	Test_R2|계수 절대값 합|내용|
|---|---|---|---|
|0.001|	0.7994|	102172.5908|규제가 가장 약한 설정의 값|
|0.01|	0.7994|	102172.4637|앞서 규제가 가장 약했던 설정의 값과 계수가 일치|
|0.1|	0.7994|	102171.0968|규제는 지속적으로 커졌지만 여전히 가장 약했던 |설정의 계수와 거의 일치|
|1|	0.7993|	102149.8317|계수 합이 조금 감소했으며 R2값이 낮아짐|
|10|	0.7991|	101725.4315|계수 축소가 뚜렷하며, R2이 낮아짐|
|100|	0.7956|	97390.1202|이번 후보 중 계수의 합이 가장 낮으며 R2 값도 가장 낮음|

<br>

> 규제를 강화하면서 계수의 전체 크기가 점차 감소했으나    
> Test의 성능이 나아지지 않았다.


### Q1. alpha가 커질수록 Ridge 계수는 일반적으로 어떻게 변하나요?

**답변:**  
- 일반적으로 계수의 절댓값이 작아진다.
- Ridge는 모든 계수를 한번에 0으로 바꾸지 않고 0에 가까운 방향으로 축소하려고 한다.


### Q2. alpha가 너무 크면 어떤 문제가 발생할 수 있나요?

**답변:**  
- 중요한 특성의 계수를 지나치게 줄여서 학습에 충분한 힌트를 얻지 못할 수 있다.
- 즉, 과소적합이 발생할 수 있다. 

---

# 필수 2. Lasso와 ElasticNet의 계수 변화

## 문제 1. Lasso에서 0이 되는 계수를 확인하세요.

### 배경

Lasso는 **L1 규제**를 사용하며, 중요하지 않은 특성의 계수를 정확히 0으로 만들 수 있습니다.

이번에는 규제 강도를 충분히 높여 Lasso의 변수 선택 효과를 직접 확인합니다.

### 요구사항

1. `Lasso(alpha=1000, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. 각 특성의 계수를 DataFrame으로 만듭니다.
4. 계수가 0인 특성의 개수를 계산합니다.
5. Test R²를 계산합니다.

### 확인 포인트

- 일부 계수가 정확히 0이 되는가?
- Ridge와 달리 Lasso가 변수 선택 효과를 갖는 이유를 이해했는가?

In [3]:
# TODO
# Lasso(alpha=1000)를 학습하고 계수와 0 계수 개수를 확인하세요.


# 1. Lasso 모델 생성
model = Lasso(alpha=1000, max_iter=10000)

# 2. 모델 학습
model.fit(X_train_scaled, y_train)

# 3. 특성별 계수 DataFrame 생성
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
})

# 4. 계수가 0인 특성 개수 계산
zero_count = (model.coef_ == 0).sum()

# 5. Test 예측 및 R² 계산
y_pred = model.predict(X_test_scaled)

test_r2 = r2_score(y_test, y_pred)

# 6. 결과 출력
display("[ Lasso Coefficients ]")
display(coef_df)

print("\nZero Coefficient Count:", zero_count)
print("Test R²:", round(test_r2, 4))

'[ Lasso Coefficients ]'

,Feature,Coefficient
0,OverallQual,25520.693859
1,GrLivArea,21419.631042
2,GarageCars,6820.209072
3,GarageArea,4370.090292
4,TotalBsmtSF,7305.573442
5,1stFlrSF,3521.096689
6,2ndFlrSF,0.000000
7,FullBath,-0.000000
8,TotRmsAbvGrd,83.200163
9,YearBuilt,6966.923170



Zero Coefficient Count: 2
Test R²: 0.796


결과
|Feature|	Coefficient|내용|
|---|---|---|
|OverallQual(품질)|25520.69| 품질이 높을 수록 예측 가격이 높아지는 방향, 그 강도|
|GrLivArea(지상면적)|21419.63| 지상 면적의 정보도 예측 가격에 양의 계수로 반영되고 있다.|
|2ndFlrSF(2층 면적)|0.00| 이 모델에서 이 2층 면적의 직접적인 기여도는 0이다.|
|FullBath(욕실)|-0.00|  이모델에서 욕실은 예측가격에 직접적인 기여도가 0이다.|

- 전체 12개의 특성 중 2개의 특성이 0의 특징을 보임
- 따라서 이 두 열의 값이 바뀌거나 제거되어도 나머지 10개의 열이 같다면 모델의 결과는 달라지지 않음

> **계수 0은 모델링할 때 삭제해도 무방할까?**   
> **2층 면적이나 욕실 수는 계수가 없어서 주택 가격과 관련이 없다는거 아니냐?**   
> 데이터 분할, 규제 강도의 변화, X의 변화가 있는 경우 계수가 0이 아닐 수도 있음

---

### Q3. Lasso에서 계수가 0이 된다는 것은 어떤 의미인가요?

**답변:**  
- 해당 특성(X)가 최종 예측식에서 영향을 주지 않는다는 의미
- Lasso의 특징은 중요도가 낮다고 판단한 특성을 자동으로 모델의 힌트로 사용하지 않음(제거하는 효과)


## 문제 2. ElasticNet을 학습하고 L1/L2 혼합을 확인하세요.

### 배경

ElasticNet은 L1과 L2 규제를 함께 사용합니다.

`l1_ratio`는 두 규제의 비율을 조절합니다.

- `l1_ratio`가 1에 가까움 → Lasso 성격이 강함
- `l1_ratio`가 0에 가까움 → Ridge 성격이 강함

### 요구사항

1. `ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test R²를 계산합니다.
4. 각 특성의 계수를 확인합니다.

### 확인 포인트

- `alpha`와 `l1_ratio`의 역할을 구분할 수 있는가?
- ElasticNet이 Lasso와 Ridge의 성격을 함께 가진다는 점을 설명할 수 있는가?

In [4]:
# TODO
# ElasticNet(alpha=0.1, l1_ratio=0.5)을 학습하세요.


# 1. ElasticNet 모델 생성
model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=10000
)

# 2. 모델 학습
model.fit(X_train_scaled, y_train)

# 3. Test 예측 및 R² 계산
y_pred = model.predict(X_test_scaled)

test_r2 = r2_score(y_test, y_pred)

# 4. 특성별 계수 DataFrame 생성
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
})

# 5. 계수가 0인 특성 개수 계산
zero_count = (model.coef_ == 0).sum()

# 6. 결과 출력
print("[ ElasticNet Coefficients ]")
display(coef_df)

print("\nZero Coefficient Count:", zero_count)
print("Test R²:", round(test_r2, 4))


[ ElasticNet Coefficients ]


,Feature,Coefficient
0,OverallQual,23514.629015
1,GrLivArea,12670.768121
2,GarageCars,6785.127803
3,GarageArea,5137.437024
4,TotalBsmtSF,8016.089858
5,1stFlrSF,9237.815373
6,2ndFlrSF,6873.818086
7,FullBath,-1895.877032
8,TotRmsAbvGrd,2541.484569
9,YearBuilt,8072.285509



Zero Coefficient Count: 0
Test R²: 0.7973


- `l1_ratio` = 0.5
    - L1항의 앞에 0.05, L2제곱합 앞에는 0.025, r=0.5라는 의미는 
    - 두 규제의 혼합비율을 중간으로 쓰겠다는 의미


|Feature|	Coefficient|내용|
|---|---|---|
|OverallQual(품질)|	23514.63 |양의 계수 중 가장 큰 값, 품질 정보가 가격예측에 가장 크게 반영|
|GrLivArea(지상면적)|	12670.77| 양의 계수 중 두번째로 큰 값, 면적도 가격 예측에 크게 반영|
|2ndFlrSF(2층 면적)|	6873.82| Lasso에서는 0이였는데 현재에서는 가격예측에 기여|
|FullBath(욕실)|	-1895.88|다른 입력이 같다면 가격예측에 음의 영향을 보인다.|

---

### Q4. `alpha`와 `l1_ratio`는 각각 무엇을 조절하나요?

**답변:**  
- `alpha` : 전체 규제의 강도를 조절
- `l1_ratio` : 규제 안에서 L1과 L2를 어떤 비율로 사용할지 조절
    - -> `l1_ratio`가 1에 가까워질수록 Lasso, 0에 가까워질수록 Ridge에 성향이 강해진다.


---

# 과제 1. 다중공선성 진단과 Ridge / Lasso / ElasticNet 비교

## 배경

같은 Train/Test 데이터에서 세 규제 모델을 비교합니다. 성취도 평가항목에 맞춰, 비교 전에 **다중공선성의 계수 해석 영향과 VIF 진단·처치 선택**을 추가합니다.

다중공선성은 입력 특성들이 서로 강한 선형 관계를 갖는 상황입니다. VIF는 한 특성을 나머지 특성으로 설명할 때 구한 결정계수 R²를 이용해 `1 / (1 - R²)`로 계산합니다. 이때 예측 대상은 `SalePrice`가 아니라 **진단할 입력 특성 하나**이며, 보조 회귀에는 절편을 포함합니다.

## 요구사항

### A. 다중공선성 진단과 처치 방향 선택

1. **Train의 입력 특성만** 이용해 특성별 VIF를 계산하고 내림차순 표 `vif_df`를 출력합니다. 기존 `X_train_scaled`를 사용해도 됩니다. 표준화는 특성 간 선형 관계 자체를 제거하지 않습니다.
2. 이번 과제에서는 `VIF < 5`: 상대적으로 낮음, `5 ≤ VIF < 10`: 주의, `VIF ≥ 10`: 높은 수준으로 분류합니다. 이는 절대적 삭제 규칙이 아닌 실습용 판단 기준입니다.
3. 주의·높은 수준인 특성을 특성명과 VIF로 제시합니다. 없으면 없다고 작성합니다. 완전한 선형 중복으로 VIF가 무한대인 경우 높은 수준으로 해석합니다. 상수 특성의 VIF는 정의하기 어려우므로 별도로 표시합니다.
4. 다중공선성이 계수의 크기·부호·안정성과 개별 변수 해석에 미치는 영향을 설명합니다.
5. 특성의 의미와 VIF 결과를 근거로 **변수 제거·결합·규제 중 한 가지 처치 방향**을 선택하고 대상·이유·한계를 설명합니다. 높은 VIF가 없으면 불필요한 제거를 하지 않는 이유와 기존 규제 비교의 목적을 설명해도 됩니다. 제거·결합의 추가 구현은 요구하지 않습니다.

### B. 기존 규제 모델 비교

다음 설정을 그대로 사용합니다.

```python
Ridge(alpha=1.0)
Lasso(alpha=1000, max_iter=10000)
ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
```

1. 세 모델을 동일한 표준화 Train 데이터로 학습합니다.
2. 동일한 Test 데이터의 R²를 계산합니다.
3. 전체 계수 크기 `abs(coef_).sum()`과 계수가 정확히 0인 개수를 계산합니다.
4. 결과를 `comparison_df`로 정리하고, 특성별 계수를 `task_coef_df`로 출력합니다.
5. Test R²와 계수 처리 차이를 함께 설명하고 Q5~Q9에 답합니다.

> Test 점수는 지정된 세 설정의 결과를 비교·해석하는 데 사용합니다. 이 점수를 보고 변수를 제거하거나 alpha를 반복 조정하지 않습니다. 실제 모델 선택·튜닝에는 Train 내부 검증 또는 CV가 필요합니다.

### 확인 포인트

- Train 입력 특성만 사용하여 VIF를 계산했는가?
- 기준과 실제 수치를 근거로 주의할 특성을 판단했는가?
- 다중공선성의 계수 해석 영향과 처치 선택 이유를 설명했는가?
- 규제가 입력 특성의 VIF 자체를 낮추는 것은 아님을 구분했는가?
- 같은 데이터에서 세 모델의 Test R²·계수 크기·0 계수 개수를 비교했는가?


In [5]:
# TODO
# A. Train 특성별 VIF 계산 및 판단
# B. 기존 세 규제 모델 비교 및 특성별 계수 표 출력
# Q5~Q9 답변 작성

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm


# A
# 절편 추가
X_vif = sm.add_constant(X_train, has_constant="add")

vif_df = pd.DataFrame({
    "Feature" : X_train.columns,
    "VIF" : [
    variance_inflation_factor(X_vif, i) 
    for i in range(1, X_vif.shape[1])
    ]
}).sort_values("VIF")


def vif_desc(value):
    if value >= 10:
        return "높은 수준"
    elif value >= 5:
        return "주의"
    else:
        return "상대적으로 낮음"

vif_df["level"] = vif_df["VIF"].apply(vif_desc)

display(vif_df[(vif_df['level']=="주의") | (vif_df["level"]=="높은 수준")])

# B
# 1. 모델 정의
models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1000, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

# 2. 결과 저장
comparison = []

coef_df = pd.DataFrame({
    "feature": X_train.columns
})

# 3. 모델별 학습 및 평가
for name, model in models.items():

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    test_r2 = r2_score(y_test, y_pred)

    coef_sum = np.abs(model.coef_).sum()

    zero_count = (model.coef_ == 0).sum()

    comparison.append({
        "model": name,
        "test_r2": test_r2,
        "abs_coef_sum": coef_sum,
        "zero_coef_count": zero_count
    })

    coef_df[name] = model.coef_

# 4. 비교 결과표
comparison_df = pd.DataFrame(comparison)

print("[ 모델별 성능 비교 ]")
display(comparison_df.round(4))

print("[ 특성별 회귀계수 ]")
display(coef_df.round(4))

,Feature,VIF,level
3,GarageArea,5.032498,주의
2,GarageCars,5.290892,주의
5,1stFlrSF,73.585999,높은 수준
6,2ndFlrSF,91.620664,높은 수준
1,GrLivArea,128.776832,높은 수준


[ 모델별 성능 비교 ]


,model,test_r2,abs_coef_sum,zero_coef_count
0,Ridge,0.7993,102149.8317,0
1,Lasso,0.7960,88648.0061,2
2,ElasticNet,0.7973,99258.8443,0


[ 특성별 회귀계수 ]


,feature,Ridge,Lasso,ElasticNet
0,OverallQual,25310.8947,25520.6939,23514.6290
1,GrLivArea,13960.7076,21419.6310,12670.7681
2,GarageCars,7041.5434,6820.2091,6785.1278
3,GarageArea,4382.9575,4370.0903,5137.4370
4,TotalBsmtSF,7238.5904,7305.5734,8016.0899
5,1stFlrSF,10011.0192,3521.0967,9237.8154
6,2ndFlrSF,7098.8967,0.0000,6873.8181
7,FullBath,-3154.7919,-0.0000,-1895.8770
8,TotRmsAbvGrd,1602.1257,83.2002,2541.4846
9,YearBuilt,8343.1173,6966.9232,8072.2855


### Q5. Ridge와 Lasso의 가장 큰 계수 처리 차이는 무엇인가요?

**답변:**
- Ridge는 L2규제로 각 계수를 제곱한 뒤 모두 더하여 penalty를 계산한다.

- Lasso는 L1규제로 각 계수의 절댓값을 모두 더하여 계산하다. 일부 계수를 정확히 0으로 만들 수 있어, 피처 선택을 고려할 수 있다.

### Q6. 규제 모델의 성능을 비교할 때 Test R²뿐 아니라 계수도 함께 보는 이유는 무엇인가요?

**답변:**
- Test R²는 모델의 예측 성능을 나타내며,

- 계수는 규제가 각 피처에 어떤 영향을 주었는지 수치적으로 보여준다.


### Q7. 다중공선성은 회귀계수의 크기·부호·안정성과 개별 변수의 영향 해석에 어떤 문제를 일으킬 수 있나요?

**답변:**
- 다중공선성은 여러개의 독립변수 사이에 강한 선형관계가 존재하는 현상이다.

- 이로인해 회귀계수의 크가와 부호가 불안정해질 수 있으며, 개별 변수가 종속변수에 미치는 영향을 구분하고 해석하기가 어려워진다.

### Q8. VIF 판단 기준은 무엇이며, 계산 결과에서 주의하거나 처치를 검토할 특성은 무엇인가요? 특성명과 VIF를 근거로 답하세요.

**답변:**
- VIF가 5 미만이면 상대적으로 낮음, 5 이상 10 미만이면 주의, 10 이상이면 높은 수준으로 판단한다.

- `GrLivArea`(128.78), `2ndFlrSF`(91.62), `1stFlrSF`(73.59)는 높은 수준으로 나타났으며, 

- `GarageCars`(5.29), `GarageArea`(5.03)는 주의 수준으로 나타났다.

### Q9. VIF 결과와 특성의 의미를 고려하여 변수 제거·결합·규제 중 어떤 처치 방법을 선택하겠나요? 구체적인 대상, 선택 이유와 한계를 설명하세요.

**답변:**
- `GrLivArea`, `1stFlrSF`, `2ndFlrSF`는 모두 주택 면적과 관련된 특성으로, 변수 제거보다는 Ridge 규제를 선택해본다.

- Ridge는 회귀계수의 크기를 조절하기 때문에 피처를 유지하면서 다중공선성으로 인해 발생한 계수의 불안정성을 완화할 수 있다.

- 하지만 다중공선성 자체를 제거하지는 못하기 때문에 개별 변수가 종속변수에 미치는 영향을 해석하는 데 한계가 있습니다.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 규제는 손실함수에 무엇을 추가하는 방법인가요?
    - 손실함수에 계수의 크기에 대한 벌점을 추가한다.

2. alpha가 커지면 규제 강도는 어떻게 변하나요?
    - 규제의 강도가 강해진다.

3. Ridge의 계수는 일반적으로 어떻게 변하나요?
    - 모든 계수를 0에 가까운 방향으로 축소함, 단 0으로 만들지는 않음

4. Lasso가 변수 선택 효과를 가지는 이유는 무엇인가요?
    - 일부 계수를 0으로 만들 수 있다.

5. ElasticNet의 `l1_ratio=0.5`는 어떤 의미인가요?
    - L1과 L2규제를 절반씩 섞는 설정

6. 규제 전에 특성 스케일을 맞추는 이유는 무엇인가요?
    - 규제는 계수 크기를 기준으로 패널티를 부여하기 때문에 특성들의 값의 범위가 다르면 패널티가 다르게 반영될 수 있음
    - 따라서 특성 스케일을 맞춘 뒤 규제를 적용한다.